# SVG Kaggle Submission Notebook

This notebook is separate from `main.ipynb` and is only for LoRA-backed inference and CSV export. It is designed for a Colab GPU runtime.

Run the setup cell first. It will:
- mount Google Drive
- clone this repo into `/content/svg-kaggle-comptetition`
- copy `test.csv` and `sample_submission.csv` from Drive into the cloned checkout if needed
- set `SVG_KAGGLE_REPO_ROOT` to the `/content` checkout path

The main notebook will then:
- load `Qwen/Qwen2.5-Coder-1.5B-Instruct` plus the saved LoRA adapter from this repo
- fail immediately unless CUDA/GPU is available
- replace any row that fails generation or SVG validation with a deterministic zero-score SVG
- only write `submission.csv` after all 1000 rows complete successfully

TPU and CPU runtimes are not supported by this notebook. In Colab, switch to `Runtime > Change runtime type > GPU` before running.


In [1]:
%cd /content/drive/MyDrive
!git clone https://github.com/Demetri65/svg-kaggle-competition-.git svg-kaggle-comptetition

/content/drive/MyDrive
fatal: destination path 'svg-kaggle-comptetition' already exists and is not an empty directory.


In [2]:
import os
import shutil
import subprocess
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

GIT_REPO_URL = "https://github.com/Demetri65/svg-kaggle-competition-.git"
CHECKOUT_PATH = Path("/content/svg-kaggle-comptetition")
DRIVE_ROOT = Path("/content/drive/MyDrive")
os.environ["SVG_KAGGLE_REPO_ROOT"] = str(CHECKOUT_PATH)


def run_command(cmd: list[str]) -> None:
    print("+", " ".join(cmd))
    subprocess.check_call(cmd)


if CHECKOUT_PATH.exists():
    shutil.rmtree(CHECKOUT_PATH)
run_command(["git", "clone", GIT_REPO_URL, str(CHECKOUT_PATH)])


def copy_required_file(name: str) -> Path:
    destination = CHECKOUT_PATH / name
    if destination.exists():
        return destination

    preferred_sources = [
        DRIVE_ROOT / "svg-kaggle-comptetition" / name,
        DRIVE_ROOT / "Colab Notebooks" / "svg-kaggle-comptetition" / name,
    ]
    for candidate in preferred_sources:
        if candidate.exists():
            shutil.copy2(candidate, destination)
            return destination

    for candidate in DRIVE_ROOT.rglob(name):
        if candidate.is_file():
            shutil.copy2(candidate, destination)
            return destination

    raise FileNotFoundError(
        f"Could not find {name} in Google Drive. Copy it into {CHECKOUT_PATH} manually and rerun this cell."
    )


for required_name in ["test.csv", "sample_submission.csv"]:
    copied_path = copy_required_file(required_name)
    print(f"Ready: {copied_path}")

print(f"SVG_KAGGLE_REPO_ROOT={os.environ['SVG_KAGGLE_REPO_ROOT']}")
print(f"Checkout exists: {CHECKOUT_PATH.exists()}")
print(f"Adapter exists: {(CHECKOUT_PATH / 'svg-lora-adapter').exists()}")
print(f"test.csv exists: {(CHECKOUT_PATH / 'test.csv').exists()}")
print(f"sample_submission.csv exists: {(CHECKOUT_PATH / 'sample_submission.csv').exists()}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
+ git clone https://github.com/Demetri65/svg-kaggle-competition-.git /content/svg-kaggle-comptetition
Ready: /content/svg-kaggle-comptetition/test.csv
Ready: /content/svg-kaggle-comptetition/sample_submission.csv
SVG_KAGGLE_REPO_ROOT=/content/svg-kaggle-comptetition
Checkout exists: True
Adapter exists: True
test.csv exists: True
sample_submission.csv exists: True


In [3]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "torch": "torch",
    "transformers": "transformers",
    "peft": "peft",
    "accelerate": "accelerate",
}

missing_packages = [
    package_name
    for module_name, package_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print(f"Installing missing packages: {missing_packages}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
else:
    print("All required packages are already installed.")

if importlib.util.find_spec("bitsandbytes") is None:
    print("bitsandbytes is not installed. CUDA-only 4-bit loading will be skipped.")
else:
    print("bitsandbytes is available for CUDA runtimes.")


All required packages are already installed.
bitsandbytes is not installed. CUDA-only 4-bit loading will be skipped.


In [4]:
import os
from pathlib import Path

REPO_ROOT = Path(os.environ.get("SVG_KAGGLE_REPO_ROOT", "")).expanduser().resolve()
required_repo_items = ["svg-lora-adapter", "test.csv", "sample_submission.csv"]
missing_repo_items = [name for name in required_repo_items if not (REPO_ROOT / name).exists()]
if not REPO_ROOT.exists() or missing_repo_items:
    raise RuntimeError(
        "SVG_KAGGLE_REPO_ROOT must point to the cloned repo in the active Colab runtime and contain "
        "svg-lora-adapter/, test.csv, and sample_submission.csv. "
        f"Current value: {REPO_ROOT} missing={missing_repo_items}"
    )

OUTPUT_ROOT = Path("/content/drive/MyDrive/svg-kaggle-comptetition/submission_outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Using repo root: {REPO_ROOT}")
print(f"Output directory ready: {OUTPUT_ROOT}")


Using repo root: /content/svg-kaggle-comptetition
Output directory ready: /content/drive/MyDrive/svg-kaggle-comptetition/submission_outputs


In [5]:
import gc
import importlib.util
import json
import os
import re
import subprocess
import sys
import textwrap
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd
import torch
from IPython.display import SVG, Markdown, display
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

os.environ["TOKENIZERS_PARALLELISM"] = "false"

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
SVG_NS = "http://www.w3.org/2000/svg"
ZERO_SCORE_SVG = f'<svg xmlns="{SVG_NS}" width="256" height="256"></svg>'
SUBMISSION_PATH = OUTPUT_ROOT / "submission.csv"
DIAGNOSTICS_PATH = OUTPUT_ROOT / "submission_diagnostics.csv"
PROGRESS_PATH = OUTPUT_ROOT / "submission_progress.csv"
CHECKPOINT_EVERY = 100
PREVIEW_COUNT = 0
MAX_NEW_TOKENS = 1024
MAX_RETRIES = 2
RETRY_TEMPERATURE = 0.2
RETRY_TOP_P = 0.9
MAX_REPAIR_CONTEXT_CHARS = 2500
SVG_CONSTRAINTS_TEXT = textwrap.dedent(
    """
    SVG constraints:
    - Return only SVG markup that begins with <svg
    - Output must be valid XML
    - width=\"256\" height=\"256\" viewBox=\"0 0 256 256\"
    - Maximum 8000 characters
    - Maximum 256 <path> elements
    """
).strip()
SYSTEM_PROMPT = (
    "You are an expert SVG generator for a Kaggle competition. "
    "Return only valid SVG markup. "
    "Do not include markdown, explanations, backticks, or any extra text. "
    "The SVG must be valid XML beginning with <svg. "
    "Use width='256' height='256' viewBox='0 0 256 256'. "
    "Keep the SVG under 8000 characters and at or under 256 path elements. "
    "Prefer compact SVGs with as few paths as possible."
)
ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse",
    "line", "polyline", "polygon", "defs", "use",
    "symbol", "clipPath", "mask", "linearGradient",
    "radialGradient", "stop", "text", "tspan",
    "title", "desc", "style", "pattern", "marker", "filter"
}
CANONICAL_TAGS = {tag.lower(): tag for tag in ALLOWED_TAGS}
URL_PATTERN = re.compile(r"(?:https?:)?//|javascript:|data:", re.IGNORECASE)
SVG_BLOCK_PATTERN = re.compile(r"<svg\b[\s\S]*?</svg>", re.IGNORECASE)
BITSANDBYTES_AVAILABLE = importlib.util.find_spec("bitsandbytes") is not None


def candidate_paths(relative_name: str):
    cwd = Path.cwd().resolve()
    seen = set()
    for base in [REPO_ROOT, cwd, *cwd.parents]:
        candidate = (base / relative_name).resolve()
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        yield candidate


def resolve_existing_path(label: str, relative_name: str, *, want_dir: bool) -> Path:
    basename = Path(relative_name).name

    for candidate in candidate_paths(relative_name):
        if candidate.exists() and ((want_dir and candidate.is_dir()) or (not want_dir and candidate.is_file())):
            return candidate

    search_roots = []
    seen_roots = set()
    cwd = Path.cwd().resolve()
    for root in [REPO_ROOT, cwd, *cwd.parents[:2]]:
        try:
            resolved_root = root.resolve()
        except Exception:
            continue
        key = str(resolved_root)
        if key in seen_roots or not resolved_root.exists() or not resolved_root.is_dir():
            continue
        seen_roots.add(key)
        search_roots.append(resolved_root)

    for root in search_roots:
        try:
            for match in root.rglob(basename):
                if want_dir and match.is_dir() and match.name == basename:
                    return match.resolve()
                if not want_dir and match.is_file() and match.name == basename:
                    return match.resolve()
        except Exception:
            continue

    repo_items = []
    try:
        repo_items = sorted(item.name for item in REPO_ROOT.iterdir())[:20]
    except Exception:
        repo_items = []

    raise FileNotFoundError(
        f"Missing {label}. Expected near {REPO_ROOT / relative_name}. "
        f"cwd={Path.cwd().resolve()} repo_root={REPO_ROOT} repo_root_exists={REPO_ROOT.exists()} repo_root_items={repo_items}"
    )


def release_runtime_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def preferred_runtime_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


ADAPTER_PATH = resolve_existing_path("svg-lora-adapter", "svg-lora-adapter", want_dir=True)
TEST_CSV_PATH = resolve_existing_path("test.csv", "test.csv", want_dir=False)
SAMPLE_SUBMISSION_PATH = resolve_existing_path("sample_submission.csv", "sample_submission.csv", want_dir=False)
ADAPTER_CONFIG_PATH = ADAPTER_PATH / "adapter_config.json"
ADAPTER_MODEL_PATH = ADAPTER_PATH / "adapter_model.safetensors"

assert ADAPTER_CONFIG_PATH.exists(), f"Missing adapter config: {ADAPTER_CONFIG_PATH}"
assert ADAPTER_MODEL_PATH.exists(), f"Missing adapter weights: {ADAPTER_MODEL_PATH}"
adapter_config = json.loads(ADAPTER_CONFIG_PATH.read_text())
assert adapter_config.get("base_model_name_or_path") == MODEL_ID, (
    f"Adapter base model mismatch: {adapter_config.get('base_model_name_or_path')} != {MODEL_ID}"
)
assert adapter_config.get("peft_type") == "LORA", f"Unexpected PEFT type: {adapter_config.get('peft_type')}"

print(f"Resolved adapter path: {ADAPTER_PATH}")
print(f"Resolved test.csv path: {TEST_CSV_PATH}")
print(f"Resolved sample_submission.csv path: {SAMPLE_SUBMISSION_PATH}")
print(f"Submission CSV path: {SUBMISSION_PATH}")
print(f"Diagnostics CSV path: {DIAGNOSTICS_PATH}")
print(f"Progress CSV path: {PROGRESS_PATH}")
print(f"Preferred runtime device: {preferred_runtime_device()}")


Resolved adapter path: /content/svg-kaggle-comptetition/svg-lora-adapter
Resolved test.csv path: /content/svg-kaggle-comptetition/test.csv
Resolved sample_submission.csv path: /content/svg-kaggle-comptetition/sample_submission.csv
Submission CSV path: /content/drive/MyDrive/svg-kaggle-comptetition/submission_outputs/submission.csv
Diagnostics CSV path: /content/drive/MyDrive/svg-kaggle-comptetition/submission_outputs/submission_diagnostics.csv
Progress CSV path: /content/drive/MyDrive/svg-kaggle-comptetition/submission_outputs/submission_progress.csv
Preferred runtime device: cuda


In [6]:
test_df = pd.read_csv(TEST_CSV_PATH, keep_default_na=False)
sample_submission_df = pd.read_csv(SAMPLE_SUBMISSION_PATH, keep_default_na=False)

assert list(test_df.columns) == ["id", "prompt"], f"Unexpected test.csv columns: {list(test_df.columns)}"
assert len(test_df) == 1000, f"Expected 1000 test rows, found {len(test_df)}"
assert list(sample_submission_df.columns) == ["id", "svg"], (
    f"Unexpected sample_submission.csv columns: {list(sample_submission_df.columns)}"
)
assert len(sample_submission_df) == len(test_df), (
    f"Expected sample_submission.csv to have {len(test_df)} rows, found {len(sample_submission_df)}"
)
assert sample_submission_df["id"].tolist() == test_df["id"].tolist(), (
    "sample_submission.csv IDs do not match test.csv order."
)

DIAGNOSTIC_COLUMNS = [
    "id",
    "prompt",
    "raw_text",
    "clean_svg",
    "cleanup_applied",
    "char_len",
    "path_count",
    "valid_final",
    "error_reason",
]

BOOL_COLUMNS = ["cleanup_applied", "valid_final"]
INT_COLUMNS = ["char_len", "path_count"]
TEXT_COLUMNS = ["id", "prompt", "raw_text", "clean_svg", "error_reason"]


def empty_diagnostics_df() -> pd.DataFrame:
    return pd.DataFrame(columns=DIAGNOSTIC_COLUMNS)


def coerce_bool(value) -> bool:
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return False
    return str(value).strip().lower() in {"1", "true", "yes", "y"}


def load_progress(progress_path: Path) -> pd.DataFrame:
    if not progress_path.exists():
        return empty_diagnostics_df()

    progress_df = pd.read_csv(progress_path, keep_default_na=False)
    if "clean_svg" not in progress_df.columns and "svg" in progress_df.columns:
        progress_df["clean_svg"] = progress_df["svg"]
    if "valid_final" not in progress_df.columns:
        source_column = "clean_svg" if "clean_svg" in progress_df.columns else "svg" if "svg" in progress_df.columns else None
        if source_column is None:
            progress_df["valid_final"] = False
        else:
            progress_df["valid_final"] = progress_df[source_column].fillna("").astype(str).str.len().gt(0)

    for column in DIAGNOSTIC_COLUMNS:
        if column not in progress_df.columns:
            progress_df[column] = "" if column in TEXT_COLUMNS else 0

    progress_df = progress_df[DIAGNOSTIC_COLUMNS].copy()
    for column in BOOL_COLUMNS:
        progress_df[column] = progress_df[column].map(coerce_bool)
    for column in INT_COLUMNS:
        progress_df[column] = pd.to_numeric(progress_df[column], errors="coerce").fillna(0).astype(int)
    for column in TEXT_COLUMNS:
        progress_df[column] = progress_df[column].fillna("").astype(str)

    progress_df = progress_df.drop_duplicates(subset=["id"], keep="last")
    progress_df = progress_df[progress_df["id"].isin(test_df["id"])].copy()
    progress_df = progress_df[progress_df["valid_final"]].copy()
    progress_df = progress_df[progress_df["clean_svg"].str.len() > 0].copy()
    progress_df = test_df[["id", "prompt"]].merge(
        progress_df.drop(columns=["prompt"], errors="ignore"),
        on="id",
        how="inner",
    )
    return progress_df[DIAGNOSTIC_COLUMNS].copy()


diagnostics_df = load_progress(PROGRESS_PATH)
print(f"Loaded {len(diagnostics_df)} completed rows from saved progress.")
display(test_df.head())
display(sample_submission_df.head())


Loaded 3 completed rows from saved progress.


,id,prompt
0,fa1d8fa7-080f-4269-a9cf-a17562c9a0ca,firewood stack cut logs wood with leaf illustr...
1,6eede943219547c22ac56085027d33cc,The image shows five horizontal lines of varyi...
2,ea045c7a247166f061ce504d9b7ccaab,A stylized icon depicting a curved arrow withi...
3,8fe82f3af89e487b31236ca829c3f071,The image contains black geometric shapes agai...
4,600464e4d92c75338462271a09b3f176,The image shows a single dark gray triangle po...


,id,svg
0,fa1d8fa7-080f-4269-a9cf-a17562c9a0ca,<svg xmlns='http://www.w3.org/2000/svg' width=...
1,6eede943219547c22ac56085027d33cc,<svg xmlns='http://www.w3.org/2000/svg' width=...
2,ea045c7a247166f061ce504d9b7ccaab,<svg xmlns='http://www.w3.org/2000/svg' width=...
3,8fe82f3af89e487b31236ca829c3f071,<svg xmlns='http://www.w3.org/2000/svg' width=...
4,600464e4d92c75338462271a09b3f176,<svg xmlns='http://www.w3.org/2000/svg' width=...


In [7]:
tokenizer = None
model = None
compute_dtype = None
RUNTIME_DEVICE = None
INFERENCE_BACKEND = None
INITIAL_RUNTIME_DEVICE = preferred_runtime_device()

if INITIAL_RUNTIME_DEVICE != "cuda":
    raise RuntimeError(
        "CUDA/GPU is required for this notebook. TPU and CPU runtimes are not supported. "
        "In Colab, switch to Runtime > Change runtime type > GPU and rerun from the top."
    )


def load_runtime_components(target_device: str):
    local_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    if local_tokenizer.pad_token is None:
        local_tokenizer.pad_token = local_tokenizer.eos_token

    local_compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    model_kwargs = {
        "trust_remote_code": True,
        "torch_dtype": local_compute_dtype,
        "device_map": "auto",
    }
    if BITSANDBYTES_AVAILABLE:
        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=local_compute_dtype,
            bnb_4bit_use_double_quant=True,
        )

    base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)
    local_model = PeftModel.from_pretrained(base_model, str(ADAPTER_PATH))
    local_model.eval()
    return local_tokenizer, local_model, local_compute_dtype


def switch_runtime(target_device: str) -> str:
    global tokenizer, model, compute_dtype, RUNTIME_DEVICE

    release_runtime_memory()
    tokenizer, model, compute_dtype = load_runtime_components(target_device)
    RUNTIME_DEVICE = target_device
    return RUNTIME_DEVICE


try:
    switch_runtime("cuda")
    INFERENCE_BACKEND = "model"
except Exception as exc:
    tokenizer = None
    model = None
    compute_dtype = None
    RUNTIME_DEVICE = None
    release_runtime_memory()
    raise RuntimeError(
        "Failed to load the LoRA-backed model on CUDA. This notebook does not permit CPU, TPU, or retrieval fallback. "
        f"Original error: {exc}"
    ) from exc

print(f"Initial preferred runtime: {INITIAL_RUNTIME_DEVICE}")
print(f"Active runtime: {RUNTIME_DEVICE}")
print(f"Inference backend: {INFERENCE_BACKEND}")
print(f"Model loaded: {MODEL_ID}")
print(f"Tokenizer vocab size: {len(tokenizer)}")
print(f"Compute dtype: {compute_dtype}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Initial preferred runtime: cuda
Active runtime: cuda
Inference backend: model
Model loaded: Qwen/Qwen2.5-Coder-1.5B-Instruct
Tokenizer vocab size: 151665
Compute dtype: torch.bfloat16


In [8]:
print(f"Outputs will be written to: {OUTPUT_ROOT}")


Outputs will be written to: /content/drive/MyDrive/svg-kaggle-comptetition/submission_outputs


In [9]:
class RowInferenceError(RuntimeError):
    def __init__(self, message: str, *, raw_text: str = ""):
        super().__init__(message)
        self.raw_text = raw_text


def local_name(name: str) -> str:
    return name.split("}", 1)[-1] if "}" in name else name


def contains_external_reference(value: str) -> bool:
    value = (value or "").strip()
    if not value:
        return False
    if URL_PATTERN.search(value):
        return True
    if value.lower().startswith("url(") and value.endswith(")"):
        inner = value[4:-1].strip().strip("'\"")
        return bool(inner) and not inner.startswith("#")
    return False


def extract_svg(text: str) -> str:
    text = (text or "").strip()
    text = re.sub(r"^```(?:xml|svg)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)
    text = re.sub(r"^<\?xml[^>]*>\s*", "", text, flags=re.IGNORECASE)
    match = SVG_BLOCK_PATTERN.search(text)
    return match.group(0).strip() if match else text.strip()


def sanitize_attributes(raw_attributes) -> dict:
    clean_attributes = {}
    for key, value in raw_attributes.items():
        attr_name = local_name(key)
        attr_value = str(value)
        lowered = attr_name.lower()

        if lowered.startswith("on"):
            raise ValueError(f"Event handler attributes are not allowed: {attr_name}")
        if lowered == "href" and attr_value.strip() and not attr_value.strip().startswith("#"):
            raise ValueError(f"External href is not allowed: {attr_value}")
        if contains_external_reference(attr_value):
            raise ValueError(f"External reference is not allowed: {attr_value}")

        clean_attributes[attr_name] = attr_value

    return clean_attributes


def sanitize_element(raw_element: ET.Element, *, is_root: bool = False) -> ET.Element:
    raw_tag = local_name(raw_element.tag)
    canonical_tag = CANONICAL_TAGS.get(raw_tag.lower())
    if canonical_tag is None:
        raise ValueError(f"Disallowed tag: {raw_tag}")

    clean_attributes = sanitize_attributes(raw_element.attrib)
    if is_root:
        preserved_root_attributes = {
            key: value
            for key, value in clean_attributes.items()
            if key not in {"xmlns", "width", "height", "viewBox"}
        }
        preserved_root_attributes.update({
            "xmlns": SVG_NS,
            "width": "256",
            "height": "256",
            "viewBox": "0 0 256 256",
        })
        clean_element = ET.Element("svg", preserved_root_attributes)
    else:
        clean_element = ET.Element(canonical_tag, clean_attributes)

    if raw_element.text:
        if canonical_tag == "style" and contains_external_reference(raw_element.text):
            raise ValueError("External references inside <style> are not allowed.")
        clean_element.text = raw_element.text

    for child in list(raw_element):
        clean_child = sanitize_element(child, is_root=False)
        clean_element.append(clean_child)
        if child.tail:
            clean_child.tail = child.tail

    return clean_element


def serialize_svg(root: ET.Element) -> str:
    svg = ET.tostring(root, encoding="unicode", short_empty_elements=True)
    svg = re.sub(r">\s+<", "><", svg).strip()
    return svg


def validate_svg(svg: str) -> dict:
    result = {
        "valid": False,
        "error_reason": "",
        "char_len": len(svg or ""),
        "path_count": 0,
    }

    if not svg:
        result["error_reason"] = "Empty SVG string."
        return result
    if not svg.startswith("<svg"):
        result["error_reason"] = "SVG must begin with <svg."
        return result
    if len(svg) > 8000:
        result["error_reason"] = "SVG exceeds 8000 characters."
        return result
    if not re.search(r"<svg\b[^>]*\bxmlns=['\"]http://www\.w3\.org/2000/svg['\"]", svg):
        result["error_reason"] = "SVG root is missing the xmlns attribute."
        return result

    try:
        root = ET.fromstring(svg)
    except Exception as exc:
        result["error_reason"] = f"Malformed XML: {exc}"
        return result

    if local_name(root.tag) != "svg":
        result["error_reason"] = "Root tag is not <svg>."
        return result
    if root.attrib.get("viewBox") != "0 0 256 256":
        result["error_reason"] = "viewBox must be exactly '0 0 256 256'."
        return result
    if root.attrib.get("width") != "256" or root.attrib.get("height") != "256":
        result["error_reason"] = "width and height must both be '256'."
        return result

    path_count = 0
    for element in root.iter():
        tag = local_name(element.tag)
        if CANONICAL_TAGS.get(tag.lower()) is None:
            result["error_reason"] = f"Disallowed tag found during validation: {tag}"
            return result
        if tag == "path":
            path_count += 1

        for key, value in element.attrib.items():
            attr_name = local_name(key)
            attr_value = str(value)
            lowered = attr_name.lower()
            if lowered.startswith("on"):
                result["error_reason"] = f"Event handler attribute found: {attr_name}"
                return result
            if lowered == "href" and attr_value.strip() and not attr_value.strip().startswith("#"):
                result["error_reason"] = f"External href found: {attr_value}"
                return result
            if contains_external_reference(attr_value):
                result["error_reason"] = f"External reference found: {attr_value}"
                return result

        if tag == "style" and contains_external_reference(element.text or ""):
            result["error_reason"] = "External references inside <style> are not allowed."
            return result

    result["path_count"] = path_count
    if path_count > 256:
        result["error_reason"] = "SVG exceeds 256 path elements."
        return result

    result["valid"] = True
    return result


def prepare_svg(raw_text: str) -> dict:
    raw_text = "" if raw_text is None else str(raw_text)
    extracted_svg = extract_svg(raw_text)
    parsed_root = ET.fromstring(extracted_svg)
    clean_root = sanitize_element(parsed_root, is_root=True)
    clean_svg = serialize_svg(clean_root)
    validation = validate_svg(clean_svg)
    if not validation["valid"]:
        raise ValueError(validation["error_reason"])

    return {
        "raw_text": raw_text,
        "clean_svg": clean_svg,
        "cleanup_applied": clean_svg.strip() != raw_text.strip(),
        "char_len": validation["char_len"],
        "path_count": validation["path_count"],
        "valid_final": True,
        "error_reason": "",
    }


def build_user_prompt(prompt: str, *, previous_output: str = "", previous_error: str = "") -> str:
    prompt_text = str(prompt).strip()
    if not previous_output:
        return textwrap.dedent(
            f"""
            Create one SVG illustration for this prompt:
            {prompt_text}

            {SVG_CONSTRAINTS_TEXT}
            """
        ).strip()

    repair_output = str(previous_output).strip()[:MAX_REPAIR_CONTEXT_CHARS]
    repair_error = str(previous_error).strip()
    return textwrap.dedent(
        f"""
        The previous SVG attempt was invalid. Rewrite it as one complete SVG that satisfies every constraint.

        Original prompt:
        {prompt_text}

        Validation error:
        {repair_error}

        Previous invalid output:
        {repair_output}

        {SVG_CONSTRAINTS_TEXT}
        """
    ).strip()


def build_model_input(prompt: str, *, previous_output: str = "", previous_error: str = "") -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": build_user_prompt(
                prompt,
                previous_output=previous_output,
                previous_error=previous_error,
            ),
        },
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def active_model_device() -> torch.device:
    return next(model.parameters()).device


def generate_raw_text(
    prompt: str,
    *,
    previous_output: str = "",
    previous_error: str = "",
    use_sampling: bool = False,
) -> str:
    if INFERENCE_BACKEND != "model" or tokenizer is None or model is None or RUNTIME_DEVICE != "cuda":
        raise RuntimeError("LoRA-backed CUDA model is not loaded.")

    input_text = build_model_input(
        prompt,
        previous_output=previous_output,
        previous_error=previous_error,
    )
    tokenized = tokenizer(input_text, return_tensors="pt", truncation=True)
    model_device = active_model_device()
    inputs = {name: tensor.to(model_device) for name, tensor in tokenized.items()}
    input_length = inputs["input_ids"].shape[1]
    generation_kwargs = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": use_sampling,
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
        "use_cache": True,
    }
    if use_sampling:
        generation_kwargs["temperature"] = RETRY_TEMPERATURE
        generation_kwargs["top_p"] = RETRY_TOP_P

    try:
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                **generation_kwargs,
            )
    except Exception as exc:
        release_runtime_memory()
        raise RuntimeError(f"Model generation failed on CUDA: {exc}") from exc

    generated_tokens = outputs[0][input_length:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True)


def infer_row(row: dict) -> dict:
    last_raw_text = ""
    attempt_errors = []

    for attempt in range(MAX_RETRIES + 1):
        raw_text = ""
        try:
            raw_text = generate_raw_text(
                row["prompt"],
                previous_output=last_raw_text,
                previous_error=attempt_errors[-1] if attempt_errors else "",
                use_sampling=attempt > 0,
            )
            prepared = prepare_svg(raw_text)
            return {
                "id": row["id"],
                "prompt": row["prompt"],
                **prepared,
            }
        except Exception as exc:
            if raw_text:
                last_raw_text = raw_text
            attempt_errors.append(f"attempt {attempt + 1}: {exc}")
            release_runtime_memory()

    raise RowInferenceError(
        f"Failed after {MAX_RETRIES + 1} attempts. {' | '.join(attempt_errors)}",
        raw_text=last_raw_text,
    )


def upsert_diagnostics(diagnostics: pd.DataFrame, record: dict) -> pd.DataFrame:
    updated = pd.concat([diagnostics, pd.DataFrame([record])], ignore_index=True)
    updated = updated.drop_duplicates(subset=["id"], keep="last")
    return updated[DIAGNOSTIC_COLUMNS].copy()


def build_ordered_diagnostics(diagnostics: pd.DataFrame) -> pd.DataFrame:
    ordered = test_df[["id", "prompt"]].merge(
        diagnostics.drop(columns=["prompt"], errors="ignore"),
        on="id",
        how="left",
    )

    for column in TEXT_COLUMNS:
        if column not in ordered.columns:
            ordered[column] = ""
        ordered[column] = ordered[column].fillna("").astype(str)
    for column in BOOL_COLUMNS:
        if column not in ordered.columns:
            ordered[column] = False
        ordered[column] = ordered[column].map(coerce_bool)
    for column in INT_COLUMNS:
        if column not in ordered.columns:
            ordered[column] = 0
        ordered[column] = pd.to_numeric(ordered[column], errors="coerce").fillna(0).astype(int)

    return ordered[DIAGNOSTIC_COLUMNS].copy()


def build_failure_record(row: dict, exc: RowInferenceError) -> dict:
    raw_text = exc.raw_text or ""
    return {
        "id": row["id"],
        "prompt": row["prompt"],
        "raw_text": raw_text,
        "clean_svg": ZERO_SCORE_SVG,
        "cleanup_applied": ZERO_SCORE_SVG.strip() != raw_text.strip(),
        "char_len": len(ZERO_SCORE_SVG),
        "path_count": 0,
        "valid_final": True,
        "error_reason": str(exc),
    }


def save_progress_artifacts(diagnostics: pd.DataFrame, *, failure_record: dict | None = None) -> None:
    ordered = build_ordered_diagnostics(diagnostics)
    completed = ordered[ordered["valid_final"] & ordered["clean_svg"].str.len().gt(0)].copy()
    completed.to_csv(PROGRESS_PATH, index=False)

    diagnostics_output = completed.copy()
    if failure_record is not None:
        diagnostics_output = pd.concat([diagnostics_output, pd.DataFrame([failure_record])], ignore_index=True)
    diagnostics_output.to_csv(DIAGNOSTICS_PATH, index=False)


def write_final_submission(diagnostics: pd.DataFrame) -> None:
    ordered = build_ordered_diagnostics(diagnostics)
    completed = ordered[ordered["valid_final"] & ordered["clean_svg"].str.len().gt(0)].copy()
    if len(completed) != len(test_df):
        raise RuntimeError(
            f"Cannot write final submission until all rows have a final SVG: {len(completed)}/{len(test_df)}"
        )

    submission_df = completed[["id", "clean_svg"]].copy()
    submission_df = submission_df.rename(columns={"clean_svg": "svg"})
    submission_df.to_csv(SUBMISSION_PATH, index=False)


def process_rows(rows_df: pd.DataFrame, diagnostics: pd.DataFrame, *, save_every: int) -> pd.DataFrame:
    if rows_df.empty:
        print("No rows to process.")
        save_progress_artifacts(diagnostics)
        return diagnostics

    rows = rows_df.to_dict(orient="records")
    for index, row in enumerate(rows, start=1):
        try:
            record = infer_row(row)
        except RowInferenceError as exc:
            failure_record = build_failure_record(row, exc)
            diagnostics = upsert_diagnostics(diagnostics, failure_record)
            save_progress_artifacts(diagnostics)
            print(f"Row {row['id']} failed generation/validation; using zero-score SVG. Reason: {exc}")
            release_runtime_memory()
            continue

        diagnostics = upsert_diagnostics(diagnostics, record)

        if index % save_every == 0 or index == len(rows):
            save_progress_artifacts(diagnostics)
            print(
                f"Saved progress after {index}/{len(rows)} new rows; "
                f"{len(diagnostics)}/{len(test_df)} rows now completed."
            )
            release_runtime_memory()

    return diagnostics


def summarize_diagnostics(diagnostics: pd.DataFrame) -> pd.DataFrame:
    ordered = build_ordered_diagnostics(diagnostics)
    summary = pd.DataFrame({
        "metric": [
            "completed_rows",
            "cleaned_rows",
            "valid_rows",
            "zero_score_rows",
            "pending_rows",
        ],
        "value": [
            int((ordered["clean_svg"].str.len() > 0).sum()),
            int(ordered["cleanup_applied"].sum()),
            int((ordered["clean_svg"].str.len() > 0).sum() - ordered["error_reason"].fillna("").ne("").sum()),
            int(ordered["error_reason"].fillna("").ne("").sum()),
            int((ordered["clean_svg"].str.len() == 0).sum()),
        ],
    })
    return summary


save_progress_artifacts(diagnostics_df)
print("Helper functions ready. Failed rows will be recorded and replaced with a zero-score SVG in the final submission.")


Helper functions ready. Failed rows will be recorded and replaced with a zero-score SVG in the final submission.


In [10]:
preview_candidates = test_df[~test_df["id"].isin(diagnostics_df["id"])].head(PREVIEW_COUNT)
print(f"Preview candidates available: {len(preview_candidates)}")

if not preview_candidates.empty:
    diagnostics_df = process_rows(preview_candidates, diagnostics_df, save_every=1)
    preview_results = build_ordered_diagnostics(diagnostics_df)
    preview_results = preview_results[preview_results["id"].isin(preview_candidates["id"])].copy()

    for row in preview_results.itertuples(index=False):
        display(Markdown(f"### Preview: `{row.id}`\n**Prompt:** {row.prompt}"))
        if row.cleanup_applied:
            raw_snippet = textwrap.shorten(row.raw_text.replace("\n", " "), width=220, placeholder="...")
            print(f"Raw model output snippet: {raw_snippet}")
        if row.error_reason:
            print(f"Zero-score SVG used: {row.error_reason}")
        display(SVG(data=row.clean_svg))
else:
    print("All preview rows have already been processed in saved progress.")

display(summarize_diagnostics(diagnostics_df))


Preview candidates available: 0
All preview rows have already been processed in saved progress.


,metric,value
0,completed_rows,3
1,cleaned_rows,3
2,valid_rows,1
3,zero_score_rows,2
4,pending_rows,997


In [ ]:
remaining_df = test_df[~test_df["id"].isin(diagnostics_df["id"])].copy()
print(f"Rows remaining for full inference: {len(remaining_df)}")

diagnostics_df = process_rows(remaining_df, diagnostics_df, save_every=CHECKPOINT_EVERY)
save_progress_artifacts(diagnostics_df)
print("Full inference pass finished.")
display(summarize_diagnostics(diagnostics_df))


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Rows remaining for full inference: 997
Row 8fe82f3af89e487b31236ca829c3f071 failed generation/validation; using zero-score SVG. Reason: Failed after 3 attempts. attempt 1: unclosed token: line 15, column 0 | attempt 2: unclosed token: line 15, column 2 | attempt 3: unclosed token: line 15, column 2
Row 5f4a048069086cff1d5f8d424af1b8ca failed generation/validation; using zero-score SVG. Reason: Failed after 3 attempts. attempt 1: unclosed token: line 39, column 0 | attempt 2: no element found: line 42, column 31 | attempt 3: no element found: line 42, column 31
Row 7e63b44725fd31b7ee98d8ba88fe29cd failed generation/validation; using zero-score SVG. Reason: Failed after 3 attempts. attempt 1: unclosed token: line 3, column 0 | attempt 2: unclosed token: line 3, column 0 | attempt 3: unclosed token: line 3, column 0


In [ ]:
ordered_diagnostics = build_ordered_diagnostics(diagnostics_df)
completed_rows = int((ordered_diagnostics["clean_svg"].str.len() > 0).sum())
assert INFERENCE_BACKEND == "model", f"Expected model inference backend, found {INFERENCE_BACKEND}."
assert tokenizer is not None and model is not None, "LoRA-backed model was not loaded."
assert RUNTIME_DEVICE == "cuda", f"Expected CUDA runtime, found {RUNTIME_DEVICE}."
assert completed_rows == len(test_df), f"Expected all {len(test_df)} rows to be completed, but only {completed_rows} are complete."
zero_score_rows = int(ordered_diagnostics["error_reason"].fillna("").ne("").sum())

write_final_submission(diagnostics_df)
final_submission = pd.read_csv(SUBMISSION_PATH, keep_default_na=False)
assert list(final_submission.columns) == ["id", "svg"], f"Unexpected submission columns: {list(final_submission.columns)}"
assert list(final_submission.columns) == list(sample_submission_df.columns), "Submission columns do not match sample_submission.csv."
assert len(final_submission) == len(test_df) == len(sample_submission_df) == 1000, (
    f"Submission row count mismatch: {len(final_submission)}"
)
assert final_submission["id"].tolist() == test_df["id"].tolist(), "Submission IDs do not match test.csv order."
assert final_submission["id"].tolist() == sample_submission_df["id"].tolist(), (
    "Submission IDs do not match sample_submission.csv order."
)
assert not final_submission.isna().any().any(), "Submission contains missing values."
assert final_submission["svg"].astype(str).str.strip().str.len().gt(0).all(), "Submission contains empty SVG values."

validation_results = final_submission["svg"].map(validate_svg).tolist()
validation_df = pd.DataFrame(validation_results)
invalid_final_rows = int((~validation_df["valid"]).sum())
assert invalid_final_rows == zero_score_rows, (
    f"Unexpected invalid SVG count in final submission: invalid_final_rows={invalid_final_rows} zero_score_rows={zero_score_rows}"
)

save_progress_artifacts(diagnostics_df)
print("Final submission assembly passed.")
print(f"submission.csv: {SUBMISSION_PATH}")
print(f"submission_diagnostics.csv: {DIAGNOSTICS_PATH}")
print(f"submission_progress.csv: {PROGRESS_PATH}")
print(f"Inference backend used: {INFERENCE_BACKEND}")
print(f"Active runtime at completion: {RUNTIME_DEVICE}")
print(f"Zero-score rows: {zero_score_rows}")

summary = summarize_diagnostics(diagnostics_df)
display(summary)

cleanup_examples = ordered_diagnostics[
    ordered_diagnostics["cleanup_applied"]
][["id", "prompt", "error_reason"]].head(5)

if not cleanup_examples.empty:
    display(Markdown("### Cleanup examples"))
    display(cleanup_examples)
else:
    print("No cleanup adjustments were needed.")
